<a href="https://colab.research.google.com/github/iannickgagnon/notebooks/blob/main/design_pattern_decorator_strategy_visitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
from __future__ import annotations

from random import randint, choice

from dataclasses import dataclass, asdict

from typing import Optional, Union, Protocol, runtime_checkable

from abc import ABC, abstractmethod

# Generic Knapsack object class

In [2]:
@dataclass
class KnapsackObject:
  """
  Generic Knapsack object class.

  Attributes:
    identifier (int): Object identifier.
    weight (int): Weight of the object.
    value (int): Value of the object.
  """
  identifier: str
  weight: int
  value: int

  def __post__init__(self):
    """
    Validate attributes after initialization.
    """
    if self.weight <= 0:
      raise ValueError("Weight must be > 0")

    if self.value < 0:
      raise ValueError("Value cannot be negative")

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Visitor pattern method for double dispatching.
    """
    raise NotImplementedError("The accept method is only implemented in the concrete subclasses.")

# Concrete KnapsackObject classes

In [3]:
class Television(KnapsackObject):
  """
  Concrete KanpsackObject class that corresponds to a television.

  Attributes:
    name (str): Name of the object.
  """
  name:str = "Television"

  def __init__(self, identifier: int, weight: int, value: int):
    """"
    Calls the parent class constructor.
    """
    super().__init__(identifier=f"{self.name} {identifier}", weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_television(self)

In [94]:
class Vase(KnapsackObject):
  """
  Concrete KnapsackObject class that corresponds to a vase.
  """
  name:str  = "Vase"

  def __init__(self, identifier: int, weight: int, value: int):
    """
    Calls the parent class constructor.
    """
    super().__init__(identifier=f"{self.name} {identifier}", weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_vase(self)

# Concrete visitors

In [4]:
@runtime_checkable
class KnapsackObjectVisitor(Protocol):
  """
  Visitor interface for KnapsackObject elements.

  Protocol defines a structural interface, which amounts to duck typing
  with type checking.

  That means:
    - A class does NOT need to inherit from `KnapsackObjectVisitor` to be
      considered a valid visitor.
    - It only needs to provide the same methods with compatible signatures.

  Example:

    class MyVisitor:
      def visit_television(self, obj: Television) -> float: ...
      def visit_vase(self, obj: Vase) -> float: ...

    `MyVisitor` is accepted anywhere a `KnapsackObjectVisitor` is expected,
    because it has the right method names/signatures.

  This is useful for the Visitor pattern because you can add new visitors
  (new operations) without modifying existing object classes, and without
  forcing visitors to inherit from a common base class.

  Normally, `Protocol` is meant for static type checkers (mypy/pyright).

  At runtime, Python does not enforce protocol conformance automatically.

  Decorating the Protocol with @runtime_checkable enables runtime checks like:

      isinstance(visitor, KnapsackObjectVisitor)

  With the decorator:
    - `isinstance` will return True if the object has the required attributes
      (here: `visit_television` and `visit_vase`) with a compatible shape.

    - It’s still not perfect "full type checking" at runtime; it’s mainly an
      structural attribute-presence check, not a guarantee about argument
      and return types.

  Without the decorator:
    - isinstance(visitor, KnapsackObjectVisitor) raises a TypeError.

  The `...` is the literal Python `Ellipsis` object.

  In a Protocol, these method bodies are stubs:
    - They say “this method must exist” and define its signature.
    - They do not provide an implementation.

  The type checker reads these stubs as abstract requirements.

  With this Protocol, the `accept()` methods can be typed as:
      def accept(self, visitor: KnapsackObjectVisitor) -> float:
          ...

  Then any new visitor class (InverseWeightVisitor, ValueDensityVisitor, etc.)
  will type-check as long as it defines the required methods with no inheritance
  required.
  """
  def visit_television(self, obj: Television) -> float: ...
  def visit_vase(self, obj: Vase) -> float: ...

In [95]:
class InverseWeightVisitor:
  """
  Returns the inverse of the object's weight.
  """
  def visit_television(self, obj: Television) -> float:
    return 1.0 / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return 1.0 / obj.weight

In [96]:
class ValueDensityVisitor:
  """
  Returns the value density (value/weight) of the object.
  """
  def visit_television(self, obj: Television) -> float:
    return obj.value / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return obj.value / obj.weight

# Data generator

In [80]:
@dataclass
class DataGenerator:
  """
  Generates a list of random objects for the Knapsack Problem.

  Attributes:
    weight_min_television_lb (int): Minimum weight of a television in pounds (lb).
    weight_max_television_lb (int): Maximum weight of a television in pounds (lb).
    value_min_television (int): Minimum value of a television in dollars.
    value_max_television (int): Maximum value of a television in dollars.
    weight_min_vase_lb (int): Minimum weight of a vase in pounds (lb).
    weight_max_vase_lb (int): Maximum weight of a vase in pounds (lb).
    value_min_vase (int): Minimum value of a vase in dollars.
    velue_max_vase (int): Maximum value of a vase in dollars.
  """
  weight_min_television_lb: int = 50
  weight_max_television_lb: int = 300
  value_min_television: int = 100
  value_max_television: int = 1500
  weight_min_vase_lb: int = 1
  weight_max_vase_lb: int = 10
  value_min_vase: int = 5
  value_max_vase: int = 3000

  def __post__init__(self):
    """
    Ensures that attributes (i.e. weights and values) are greater than zero.
    """
    accumulator: list[str] = []
    for attribute, value in asdict(self).items():
      if value <= 0:
        accumulator.append(attribute)
    if accumulator:
      raise ValueError(f"The following attributes must be greater than zero : {accumulator}")


  def generate(self, nb_objects: int = 10, verbose: bool = True) -> list[KnapsackObject]:
    """
    Generates a list of nb_objects random objects.

    Args:
      nb_objects (int, optional): The number of objects to generate. Defaults to 10.
      verbose (bool, optional): Whether to print progress or not. Defaults to True.

    Returns:
      (tuple[KnapsackObject]): A list of random KnapsackObject instances.

    Raises:
      ValueError: When the supplied number of objects is less than zero.
    """

    # Validate number of objects
    if nb_objects < 0:
      raise ValueError("The number of objects to generate must be >= 0")

    # Initialize return container
    objects: list[KnapsackObject] = []

    # Generate random objects
    for i in range(nb_objects):

      # NOTE: Type is simplified for readability
      obj_factory: tuple = choice([(Television,
                            self.weight_min_television_lb,
                            self.weight_max_television_lb,
                            self.value_min_television,
                            self.value_max_television),
                            (Vase,
                            self.weight_min_vase_lb,
                            self.weight_max_vase_lb,
                            self.value_min_vase,
                            self.value_max_vase)])

      weight_min: int = obj_factory[1]
      weight_max: int = obj_factory[2]
      value_min: int = obj_factory[3]
      value_max: int = obj_factory[4]

      random_weight: int = randint(weight_min, weight_max)
      random_value: int = randint(value_min, value_max)

      random_object: KnapsackObject = obj_factory[0](identifier=i + 1,
                                                     weight=random_weight,
                                                     value=random_value)

      objects.append(random_object)

      # Show progress
      if verbose:
        print(f"Added object no.{i + 1}: {random_object}")

    return objects

# Knapsack problem instance

In [7]:
@dataclass
class KnapsackProblem:
  """
  Contains necessary data for the KnapSackProblemSolver.

  Attributes:
    objects (list[KnapsackObject]): A list of available Objects.
    weights (list[int]): The weights of the objects.
    values (list[int]): The values of the objects.
    value_densities (list[float]): The value densities (i.e. value / weight) of the objects.
  """
  objects: list[KnapsackObject]
  weights: list[int]
  values: list[int]
  value_densities: list[float]

# Knapsack problem parser

In [104]:
class KnapsackProblemParser:
  """
  Generates a KnapsackProblem instance from data.

  Attributes:
    objects (list[KnapsackObject]): Objects to choose from.
    value_densities (list[float]): Order-matching value densities of the objects.
  """

  def __init__(self,
               objects: list[KnapsackObject],
               value_densities: list[float]):

    # Store objects and inverse weights
    self.objects = objects
    self.value_densities = value_densities

  def parse(self) -> KnapsackProblem:
    """
    Builds KnapsackProblem object from data.

    Returns:
      (KnapsackProblem): Generated problem instance.
    """

    # Extract identifiers, weights and values as lists
    weights: list[int] = [obj.weight for obj in objects]
    values: list[int] = [obj.value for obj in objects]

    # Instanciate problem instance
    knapsack_problem = KnapsackProblem(objects=objects,
                                       weights=weights,
                                       values=values,
                                       value_densities=self.value_densities)

    return knapsack_problem

# Solver strategies

In [105]:
class KnapsackSolver:
  """
  Solver for the Knapsack Problem implemented as the client
  of the strategy pattern.

  Attributes:
    __strategy (Optional[KnapsackSolverStrategy]): Interchangeable Knapsack solver strategy. Defaults to None.
  """

  def __init__(self):
    self.__strategy = None

  # Getter and setter to inspect and swap strategy dynamically
  @property
  def strategy(self) -> Optional[KnapsackSolverStrategy]:
    return self.__strategy

  @strategy.setter
  def strategy(self, new_strategy: KnapsackSolverStrategy) -> None:
    self.__strategy = new_strategy

  def solve(self, problem: KnapsackProblem, knapsack: Knapsack) -> None:
    """
    Delegate to the solving strategy.

    Attributes:
      problem (KnapsackProblem): The problem instance to solve.
      knapsack (Knapsack): The knapsack to fill.

    Raises:
      ValueError: If the solver's strategy is not set.
    """

    # Ensure strategy has been set
    if self.__strategy is None:
      raise ValueError("Must initialize solver strategy first")

    self.__strategy.solve(problem=problem, knapsack=knapsack)

In [106]:
class Knapsack:
  """
  A simple knapsack.

  Attributes:
    capacity_lb (int): The knapsack's capacity in pounds (lb).
    content (list[KnapsackObject]): The knapsack's content. Default is an empty list.
    current_weight (int): Total weight of the knapsack's content. Defaults to zero.
    total_value (int): Total value of the knapsack's content. Defaults to zero.
  """

  def __init__(self, capacity_lb: int):
    self.capacity_lb: int = capacity_lb
    self.content: list[KnapsackObject] = []
    self.total_weight: int = 0
    self.total_value: int = 0

  def does_it_fit(self, obj: KnapsackObject):
    """
    Checks if the object fits in the knapsack given its current content
    and capabity.

    Args:
      obj (KnapsackObject): An object (e.g. Television or Vase).

    Returns:
      bool: Whether it fits or not.
    """
    return self.total_weight + obj.weight <= self.capacity_lb


  def add_item(self, obj: KnapsackObject) -> None:
    """
    Stores an item in the knapsack.

    Args:
      obj (KnapsackObject): Object to be stored.

    Raises:
      ValueError: If adding the object surpasses the knapsack's capacity.
    """

    # Make sure the item fits
    total_weight: int = self.total_weight + obj.weight
    if not self.does_it_fit(obj):
      raise ValueError(f"Total capacity exceeded: {total_weight} lbs vs. {self.capacity_lb} lbs)")

    # Put object in knapack
    self.content.append(obj)

    # Update weight and value
    self.total_weight = total_weight
    self.total_value += obj.value

In [107]:
class KnapsackSolverStrategy(ABC):
  """
  Strategy pattern for our solver.
  """
  @abstractmethod
  def solve(self, problem: KnapsackProblem, knapsack: Knapsack) -> None:
    pass

In [108]:
class GreedyMaximumValueStrategy(KnapsackSolverStrategy):
  """
  Concrete KnapsackProblemSolver strategy.

  Add the items with the highest values first until either:
    1. All the items are in the knapsack.
    2. The knapsack is full.
  """

  def solve(self, problem: KnapsackProblem, knapsack: Knapsack) -> None:

    # Value given to one of the problem's objects when it is alredy in the knapsack
    INVALID_VALUE: int = -1

    # Copy the list of values to avoid modifying the problem instance
    values_proxy: list[int] = problem.values.copy()

    while True:

      # Find next most valuable object's value
      highest_value: int = max(values_proxy)

      # All items are in the bag
      if highest_value == INVALID_VALUE:
        break

      # Find next most valuable object's position and reference
      most_valuable_idx: int = values_proxy.index(highest_value)
      most_valuable_object: KnapsackObject = problem.objects[most_valuable_idx]

      # Add if it fits, leave otherwise
      if knapsack.does_it_fit(most_valuable_object):
        knapsack.add_item(most_valuable_object)

        # Remove item from proxy (i.e. remove from bag)
        values_proxy[most_valuable_idx] = INVALID_VALUE

      else:
        break

In [109]:
class GreedyMinimumWeightStrategy(KnapsackSolverStrategy):
  """
  Concrete KnapsackProblemSolver strategy.

  Add the items with the lowest weight first until either:
    1. All the items are in the knapsack.
    2. The knapsack is full.

  NOTE: This function is not fully typed to avoid Google Colab linter errors.
  """

  def solve(self, problem: KnapsackProblem, knapsack: Knapsack) -> None:

    # Value given to one of the problem's objects when it is alredy in the knapsack
    INVALID_VALUE: float = float('inf')

    # Copy the list of values to avoid modifying the problem instance
    weights_proxy: list= problem.weights.copy()

    while True:

      # Find next lightest object
      lowest_weight = min(weights_proxy)

      # All items are in the bag
      if lowest_weight == INVALID_VALUE:
        break

      # Find next lightest object's position and reference
      lightest_idx: int = weights_proxy.index(lowest_weight)
      lightest_object: KnapsackObject = problem.objects[lightest_idx]

      # Add if it fits, leave otherwise
      if knapsack.does_it_fit(lightest_object):
        knapsack.add_item(lightest_object)

        # Remove item from proxy (i.e. remove from bag)
        weights_proxy[lightest_idx] = INVALID_VALUE

      else:
        break

In [110]:
class GreedyValueDensityStrategy(KnapsackSolverStrategy):
  """
  Concrete KnapsackProblemSolver strategy.

  Add the items with the highest value density first until either:
    1. All the items are in the knapsack.
    2. The knapsack is full.

  NOTE: This function is not fully typed to avoid Google Colab linter errors.
  """

  def solve(self, problem: KnapsackProblem, knapsack: Knapsack) -> None:

    # Value given to one of the problem's objects when it is alredy in the knapsack
    INVALID_VALUE: int = -1

    # Copy the list of values to avoid modifying the problem instance
    value_densities_proxy: list= problem.value_densities.copy()

    while True:

      # Find next highest value density object
      highest_value_density = max(value_densities_proxy)

      # All items are in the bag
      if highest_value_density == INVALID_VALUE:
        break

      # Find next highes value density object's position and reference
      highest_value_density_idx: int = value_densities_proxy.index(highest_value_density)
      highest_value_density_object: KnapsackObject = problem.objects[highest_value_density_idx]

      # Add if it fits, leave otherwise
      if knapsack.does_it_fit(highest_value_density_object):
        knapsack.add_item(highest_value_density_object)

        # Remove item from proxy (i.e. remove from bag)
        value_densities_proxy[highest_value_density_idx] = INVALID_VALUE

      else:
        break

In [111]:
# Generate test data
objects = DataGenerator().generate(verbose=False)

# Initialize visitors
inverse_weight_visitor = InverseWeightVisitor()
value_density_visitor = ValueDensityVisitor()

# Client accepts, visitors vitis
inverse_weights: list[float] = []
value_densities: list[float] = []
for knapsack_obj in objects:
  inverse_weights.append(knapsack_obj.accept(inverse_weight_visitor))
  value_densities.append(knapsack_obj.accept(value_density_visitor))

# Show visitors results
print()
for weight, density in zip(inverse_weights, value_densities):
  print(f"inverse weight (1/lb) = {weight}\n"
        f"value density  ($/lb) = {density}\n")

# Initialize problem parser
parser = KnapsackProblemParser(objects=objects,
                               value_densities=value_densities)

# Initialize problem instance
problem_instance: KnapsackProblem = parser.parse()

# Initialize solver
solver = KnapsackSolver()

# Initialize and test strategies
strategies = [GreedyMaximumValueStrategy(),
              GreedyMinimumWeightStrategy(),
              GreedyValueDensityStrategy()]

for strategy in strategies:

  # A new knapsack for each strategy
  knapsack: Knapsack = Knapsack(capacity_lb=350)

  # Apply current strategy
  solver.strategy = strategy
  solver.solve(problem=problem_instance,
               knapsack=knapsack)

  # Show solution
  print(f"Solution obtained by the {strategy.__class__.__name__} strategy")

  # Show solution
  print(f"\n\tNumber of items : {len(knapsack.content)}")
  print(f"\tTotal value     : {knapsack.total_value}")

  print("\n\tKnapsack content:\n")

  for i, obj in enumerate(knapsack.content):
    print(f"\t\tObject no.{i + 1} : {obj}")
  print("\n")



inverse weight (1/lb) = 0.2
value density  ($/lb) = 90.4

inverse weight (1/lb) = 0.01694915254237288
value density  ($/lb) = 20.372881355932204

inverse weight (1/lb) = 1.0
value density  ($/lb) = 947.0

inverse weight (1/lb) = 0.2
value density  ($/lb) = 476.6

inverse weight (1/lb) = 0.0033333333333333335
value density  ($/lb) = 4.403333333333333

inverse weight (1/lb) = 0.5
value density  ($/lb) = 403.0

inverse weight (1/lb) = 0.007575757575757576
value density  ($/lb) = 1.7954545454545454

inverse weight (1/lb) = 0.1111111111111111
value density  ($/lb) = 292.44444444444446

inverse weight (1/lb) = 0.14285714285714285
value density  ($/lb) = 51.285714285714285

inverse weight (1/lb) = 0.005
value density  ($/lb) = 5.45

Solution obtained by the GreedyMaximumValueStrategy strategy

	Number of items : 3
	Total value     : 6336

	Knapsack content:

		Object no.1 : Vase(identifier='Vase 8', weight=9, value=2632)
		Object no.2 : Vase(identifier='Vase 4', weight=5, value=2383)
		Objec